# Previous Day Segment Probability Analysis — Extended Segments

Segments are defined based on the 10th–90th percentile range of the actual high/low distribution,
so boundaries align with real quartiles rather than arbitrary 0–100% equal splits.

Session: 18:00→17:00 EST (futures convention).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────────────────────
FILE_PATH  = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
START_DATE = '2013-01-01'
END_DATE   = '2025-12-31'

# Segment boundaries in % of prior day range — derived from percentile analysis:
# covers 10th–90th pctile of both high and low distributions
# -90% = P10 of lows,  +185% = P90 of highs
SEG_EDGES  = [-90, -45, 0, 50, 100, 145, 190]
SEG_LABELS = [
    '-90% to -45%  (deep low ext.)',
    '-45% to 0%    (mod. low ext.)',
    '0% to 50%     (lower half)',
    '50% to 100%   (upper half)',
    '100% to 145%  (mod. high ext.)',
    '145% to 190%  (deep high ext.)',
]
SEG_LABELS_SHORT = ['-90–-45%', '-45–0%', '0–50%', '50–100%', '100–145%', '145–190%']
N_SEGMENTS = len(SEG_EDGES) - 1
# ─────────────────────────────────────────────────────────────────────────────

print('Loading data...')
df = pd.read_csv(FILE_PATH)
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)
df = df.loc[START_DATE:END_DATE]
print(f'Loaded {len(df):,} rows  |  {df.index[0].date()} → {df.index[-1].date()}')

In [ ]:
# ── BUILD DAILY SESSIONS ──────────────────────────────────────────────────────
df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
trading_days = sorted(df['NY_Trading_Day'].unique())

records = []
for day in trading_days:
    actual_date   = day + pd.Timedelta(hours=7)
    session_start = actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
    session_end   = actual_date.replace(hour=17, minute=0, second=0)
    day_data = df.loc[session_start:session_end]
    if day_data.empty:
        continue
    records.append({
        'trading_day': day,
        'day_high':    day_data['High'].max(),
        'day_low':     day_data['Low'].min(),
        'day_range':   day_data['High'].max() - day_data['Low'].min(),
    })

daily = pd.DataFrame(records).set_index('trading_day')
print(f'Built {len(daily):,} daily sessions')

In [ ]:
# ── COMPUTE RAW PERCENTAGES ───────────────────────────────────────────────────
days_list = daily.index.tolist()
high_pcts, low_pcts = [], []

for i in range(1, len(days_list)):
    today = daily.loc[days_list[i]]
    prev  = daily.loc[days_list[i - 1]]
    if prev['day_range'] <= 0:
        continue
    high_pcts.append((today['day_high'] - prev['day_low']) / prev['day_range'] * 100)
    low_pcts.append( (today['day_low']  - prev['day_low']) / prev['day_range'] * 100)

high_pcts = np.array(high_pcts)
low_pcts  = np.array(low_pcts)
total_days = len(high_pcts)
print(f'Total day pairs: {total_days:,}')

# Quick percentile reminder
for label, arr in [('HIGH', high_pcts), ('LOW', low_pcts)]:
    p = np.percentile(arr, [10, 25, 50, 75, 90])
    print(f'{label}  P10={p[0]:.1f}%  P25={p[1]:.1f}%  P50={p[2]:.1f}%  P75={p[3]:.1f}%  P90={p[4]:.1f}%')

In [ ]:
# ── SEGMENT COUNTING ──────────────────────────────────────────────────────────
edges = np.array(SEG_EDGES, dtype=float)

def count_segments(pcts, edges):
    """Count values in each [edges[k], edges[k+1]) bucket. Last bucket closed on right."""
    n = len(edges) - 1
    counts = np.zeros(n, dtype=int)
    below  = int(np.sum(pcts < edges[0]))
    above  = int(np.sum(pcts > edges[-1]))
    for k in range(n):
        lo, hi = edges[k], edges[k + 1]
        if k == n - 1:
            counts[k] = np.sum((pcts >= lo) & (pcts <= hi))
        else:
            counts[k] = np.sum((pcts >= lo) & (pcts <  hi))
    return counts, below, above

high_counts, high_below, high_above = count_segments(high_pcts, edges)
low_counts,  low_below,  low_above  = count_segments(low_pcts,  edges)

high_pct_of_total = high_counts / total_days * 100
low_pct_of_total  = low_counts  / total_days * 100

# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
rows = []
for k in range(N_SEGMENTS):
    rows.append({
        'Segment':        SEG_LABELS[k],
        'High count':     high_counts[k],
        'High % of days': round(high_pct_of_total[k], 1),
        'Low count':      low_counts[k],
        'Low % of days':  round(low_pct_of_total[k], 1),
    })

# Outside-range rows
rows.append({'Segment': f'Below {SEG_EDGES[0]}%',
             'High count': high_below, 'High % of days': round(high_below/total_days*100,1),
             'Low count':  low_below,  'Low % of days':  round(low_below/total_days*100,1)})
rows.append({'Segment': f'Above {SEG_EDGES[-1]}%',
             'High count': high_above, 'High % of days': round(high_above/total_days*100,1),
             'Low count':  low_above,  'Low % of days':  round(low_above/total_days*100,1)})

summary = pd.DataFrame(rows).set_index('Segment')
print(summary.to_string())
print(f'\nColumn totals (should sum to ~100% each):')
print(f"  High: {summary['High % of days'].sum():.1f}%")
print(f"  Low:  {summary['Low % of days'].sum():.1f}%")

In [ ]:
# ── MAIN CHART ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(
    f'NQ Futures — Where does the day High/Low form relative to the prior day range?\n'
    f'{START_DATE[:4]}–{END_DATE[:4]}  |  Session 18:00→17:00 EST  |  n={total_days:,} days',
    fontsize=13, fontweight='bold'
)

seg_y  = np.arange(N_SEGMENTS)
colors_high = plt.cm.Blues(np.linspace(0.35, 0.85, N_SEGMENTS))
colors_low  = plt.cm.RdPu(np.linspace(0.35, 0.85, N_SEGMENTS))

for ax, pct_vals, colors, title, color_line in [
    (axes[0], high_pct_of_total, colors_high, 'Where does the DAY HIGH form?', '#2962FF'),
    (axes[1], low_pct_of_total,  colors_low,  'Where does the DAY LOW form?',  '#E91E63'),
]:
    # Plot bottom→top so seg[0] (-90–-45%) is at bottom, seg[5] (145–190%) at top
    bars = ax.barh(seg_y, pct_vals, height=0.6,
                   color=colors, edgecolor='white', linewidth=0.8)

    # Reference line: uniform distribution
    uniform = 100 / N_SEGMENTS
    ax.axvline(uniform, color='gray', linestyle='--', linewidth=1.2,
               alpha=0.7, label=f'Uniform ({uniform:.0f}%/seg)')

    # Shade the 0–100% region (inside prior day range)
    # Segments 2 and 3 (0–50%, 50–100%) are indices 2 and 3
    for idx in [2, 3]:
        ax.barh(seg_y[idx], pct_vals[idx], height=0.6,
                color=colors[idx], edgecolor='gold', linewidth=1.5)

    ax.set_yticks(seg_y)
    ax.set_yticklabels(SEG_LABELS, fontsize=9.5)
    ax.set_xlabel('% of trading days', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

    # Value labels
    for bar, val in zip(bars, pct_vals):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=9.5, fontweight='bold')

    # Annotation: prior day range boundary
    ax.annotate('← prior day range →', xy=(0.5, 0.5), xycoords='axes fraction',
                ha='center', va='center', fontsize=7.5, color='goldenrod',
                xytext=(0.5, 0.395), textcoords='axes fraction')

    ax.set_xlim(0, pct_vals.max() * 1.3 + 3)

# Gold border on the inside-range bars label
for ax in axes:
    ax.annotate('gold border = inside prior day range', xy=(0.98, 0.02),
                xycoords='axes fraction', ha='right', va='bottom',
                fontsize=7.5, color='goldenrod')

plt.tight_layout()
plt.savefig('prev_day_segment_extended.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: prev_day_segment_extended.png')

In [ ]:
# ── CONTINUOUS DISTRIBUTION with segment overlays ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(
    'Full distribution of day H/L as % of prior day range\n'
    'Segment boundaries shown as vertical lines',
    fontsize=12, fontweight='bold'
)

bin_width = 10  # 10% bins for smooth histogram
all_bins  = np.arange(-200, 350, bin_width)

seg_colors_h = plt.cm.Blues(np.linspace(0.3, 0.85, N_SEGMENTS))
seg_colors_l = plt.cm.RdPu(np.linspace(0.3, 0.85, N_SEGMENTS))

for ax, pcts, seg_colors, title in [
    (axes[0], high_pcts, seg_colors_h, 'Day HIGH as % of prior day range'),
    (axes[1], low_pcts,  seg_colors_l, 'Day LOW as % of prior day range'),
]:
    counts_all, bin_edges = np.histogram(pcts, bins=all_bins)
    pct_all = counts_all / total_days * 100

    # Colour each bar by which segment it falls in
    for cnt, lo, hi in zip(pct_all, bin_edges[:-1], bin_edges[1:]):
        mid = (lo + hi) / 2
        # Find which segment this bin midpoint belongs to
        seg_idx = None
        for k in range(N_SEGMENTS):
            if edges[k] <= mid < edges[k + 1]:
                seg_idx = k
                break
            elif k == N_SEGMENTS - 1 and mid == edges[k + 1]:
                seg_idx = k
        color = seg_colors[seg_idx] if seg_idx is not None else 'lightgray'
        alpha = 0.85 if seg_idx is not None else 0.4
        ax.bar(lo, cnt, width=bin_width * 0.92, color=color, alpha=alpha,
               edgecolor='white', linewidth=0.3, align='edge')

    # Segment boundary lines
    for k, edge in enumerate(edges):
        ax.axvline(edge, color='black', linestyle='--', linewidth=1, alpha=0.6)
        if k < N_SEGMENTS:
            mid_x = (edges[k] + edges[k + 1]) / 2
            ax.text(mid_x, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1,
                    SEG_LABELS_SHORT[k], ha='center', va='top',
                    fontsize=6.5, color='black', rotation=0)

    # Percentile markers
    for p, label in [(10,'P10'), (25,'P25'), (50,'P50'), (75,'P75'), (90,'P90')]:
        val = np.percentile(pcts, p)
        ax.axvline(val, color='red', linestyle=':', linewidth=1, alpha=0.5)
        ax.text(val, 0.5, label, fontsize=6.5, color='red', rotation=90, va='bottom')

    ax.set_xlabel('% of prior day range', fontsize=10)
    ax.set_ylabel('% of days', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlim(-200, 350)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d%%'))
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('prev_day_segment_distribution_extended.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: prev_day_segment_distribution_extended.png')

In [ ]:
# ── DAY OF WEEK BREAKDOWN ─────────────────────────────────────────────────────
days_list_dt = [d + pd.Timedelta(hours=7) for d in days_list]
dow_records  = []

for i in range(1, len(days_list)):
    today = daily.loc[days_list[i]]
    prev  = daily.loc[days_list[i - 1]]
    if prev['day_range'] <= 0:
        continue
    h_pct = (today['day_high'] - prev['day_low']) / prev['day_range'] * 100
    l_pct = (today['day_low']  - prev['day_low']) / prev['day_range'] * 100
    dow_records.append({'dow': days_list_dt[i].weekday(), 'h_pct': h_pct, 'l_pct': l_pct})

dow_df    = pd.DataFrame(dow_records)
dow_names = {0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri'}

fig, axes = plt.subplots(2, 5, figsize=(22, 10), sharey=False)
fig.suptitle('Segment probabilities by day of week\n(gold border = inside prior day range)',
             fontsize=13, fontweight='bold')

for col, (dow_num, dow_name) in enumerate(dow_names.items()):
    subset = dow_df[dow_df['dow'] == dow_num]
    n_dow  = len(subset)

    for row_idx, (pcts_col, colors, ylabel) in enumerate([
        ('h_pct', plt.cm.Blues(np.linspace(0.35, 0.85, N_SEGMENTS)),  'High % of days'),
        ('l_pct', plt.cm.RdPu(np.linspace(0.35, 0.85, N_SEGMENTS)),   'Low % of days'),
    ]):
        ax   = axes[row_idx][col]
        pcts = subset[pcts_col].values
        counts, below, above = count_segments(pcts, edges)
        pct_vals = counts / n_dow * 100

        x = np.arange(N_SEGMENTS)
        bars = ax.bar(x, pct_vals, color=colors, edgecolor='white', linewidth=0.5)

        # Gold border on inside-range segments (indices 2 and 3: 0–50%, 50–100%)
        for idx in [2, 3]:
            ax.bar(x[idx], pct_vals[idx], color=colors[idx],
                   edgecolor='gold', linewidth=2)

        # Uniform reference
        ax.axhline(100 / N_SEGMENTS, color='gray', linestyle='--',
                   linewidth=0.8, alpha=0.5)

        ax.set_xticks(x)
        ax.set_xticklabels(SEG_LABELS_SHORT, rotation=45, ha='right', fontsize=6)
        ax.set_title(f'{dow_name}  (n={n_dow})', fontsize=10, fontweight='bold')
        if col == 0:
            ax.set_ylabel(ylabel, fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False)

        for xi, v in zip(x, pct_vals):
            ax.text(xi, v + 0.3, f'{v:.0f}%', ha='center', va='bottom', fontsize=6.5)

        ax.text(0.98, 0.97,
                f'<{SEG_EDGES[0]}%: {below/n_dow*100:.0f}%\n>{SEG_EDGES[-1]}%: {above/n_dow*100:.0f}%',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=6, color='gray')

plt.tight_layout()
plt.savefig('prev_day_segment_dow_extended.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: prev_day_segment_dow_extended.png')

In [ ]:
# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print('=' * 65)
print(f'SEGMENT SUMMARY  (n={total_days:,} days, 2013–2025)')
print('=' * 65)
print(f'{"Segment":<32} {"High %":>8} {"Low %":>8}')
print('-' * 65)
for k in range(N_SEGMENTS):
    marker = ' ◄ inside prior range' if k in [2, 3] else ''
    print(f'{SEG_LABELS[k]:<32} {high_pct_of_total[k]:>7.1f}% {low_pct_of_total[k]:>7.1f}%{marker}')
print('-' * 65)
print(f'{f"Below {SEG_EDGES[0]}%":<32} {high_below/total_days*100:>7.1f}% {low_below/total_days*100:>7.1f}%')
print(f'{f"Above {SEG_EDGES[-1]}%":<32} {high_above/total_days*100:>7.1f}% {low_above/total_days*100:>7.1f}%')
print('=' * 65)
print(f'\nKey takeaways:')
print(f'  Median high: {np.median(high_pcts):.1f}% of prior day range')
print(f'  Median low:  {np.median(low_pcts):.1f}% of prior day range')
print(f'  High stays inside prior range (0–100%): {np.sum((high_pcts>=0)&(high_pcts<=100))/total_days*100:.1f}% of days')
print(f'  Low  stays inside prior range (0–100%): {np.sum((low_pcts>=0) &(low_pcts<=100))/total_days*100:.1f}% of days')